# Web Data Science
## Week 03 - XML and JSON

INFO 4871 - Fall 2024  
[Brian C. Keegan, Ph.D.](http://brianckeegan.com/)  
[Assistant Professor, Department of Information Science](https://www.colorado.edu/cmci/people/information-science/brian-c-keegan)  
University of Colorado Boulder  

Copyright and distributed under an [MIT License](https://opensource.org/licenses/MIT)

## Import libraries

In [1]:
import requests

### XML

The [Extensible Markup Language](https://www.w3.org/XML/) (XML) is a markup language for representing data structures. XML was all the rage at the turn of the century: "many software designers can barely contain their excitement over its potential to establish a real Internet lingua franca" (*The New York Times* in 2000: "[The Next Big Step? It's Called XML](https://www.nytimes.com/2000/06/07/business/the-next-big-leap-it-s-called-xml.html)"). That obviously did not come to pass. But XML remains a robust and open—though verbose—standard for representing structured data.

XML has taken on something of an afterlife as the official data standard for the U.S. Congress. The [House](http://clerk.house.gov/index.aspx) and [Senate](https://www.senate.gov/general/XML.htm) both release information about members, committees, schedules, legislation, and votes in XML. These are immaculately formatted and documented and remarkably up-to-date.

![Congress MemberData XML schema](xml.png)

Use the `requests` library to make a HTTP get request to the House's webserver and get the list of current member data.

In [2]:
house_raw = requests.get('http://clerk.house.gov/xml/lists/MemberData.xml').text


In [3]:
senate_raw = requests.get('https://www.senate.gov/legislative/LIS_MEMBER/cvc_member_data.xml').text

In [6]:
house_raw[:1000]

'<?xml version="1.0" encoding="UTF-8"?><MemberData publish-date="September 3, 2024"><title-info><congress-num>118</congress-num><congress-text>One Hundred Eighteenth Congress</congress-text><session>2</session><majority>R</majority><minority>D</minority><clerk>KEVIN F. McCUMBER</clerk><weburl>https://clerk.house.gov</weburl></title-info><members><member><statedistrict>AK00</statedistrict><member-info><namelist>Peltola, Mary</namelist><bioguideID>P000619</bioguideID><lastname>Peltola</lastname><firstname>Mary</firstname><middlename>Sattler</middlename><sort-name>PELTOLA,MARY</sort-name><suffix/><courtesy>Mrs.</courtesy><prior-congress>117</prior-congress><official-name>Mary Sattler Peltola</official-name><formal-name>Mrs. Peltola</formal-name><party>D</party><caucus>D</caucus><state postal-code="AK"><state-fullname>Alaska</state-fullname></state><district>At Large</district><townname>Bethel</townname><office-building>CHOB</office-building><office-room>153</office-room><office-zip>20515<

This data is still in a string format (`type(house_raw)`), so it's difficult to search and navigate. Let's make our first soup together using [BeautifulSoup](https://www.crummy.com/software/BeautifulSoup/bs4/doc/).

In [9]:
# First import the library
from bs4 import BeautifulSoup

# Then make the soup, specifying the "lxml" parser
house_soup = BeautifulSoup(house_raw)

/var/folders/lr/v195xr617d32k2zdwwh30sb40000gn/T/ipykernel_57760/123764704.py:5: XMLParsedAsHTMLWarning: It looks like you're parsing an XML document using an HTML parser. If this really is an HTML document (maybe it's XHTML?), you can ignore or filter this warning. If it's XML, you should know that using an XML parser will be more reliable. To parse this document as XML, make sure you have the lxml package installed, and pass the keyword argument `features="xml"` into the BeautifulSoup constructor.
  house_soup = BeautifulSoup(house_raw)


In [ ]:
# Try this first
! conda install lxml

In [ ]:
# Try this if the first doesn't work
! pip install lxml bs4

What's so great about this soup-ified string? We now have a suite of new functions and methods that let us navigate the tree. First, let's inspect the different tags/elements in this tree of House member data. This is the full tree of data.

In [14]:
# Make an empty list to store data
children = []

# Start a loop to go through all the children tags in house_soup
for tag in house_soup.findChildren():
    
    # If the name of the tag (tag.name) is not already in the children list
    if tag.name not in children:
        
        # Add the name of the tag to the children list
        children.append(tag.name)

# Look at the list members
children

['html',
 'body',
 'memberdata',
 'title-info',
 'congress-num',
 'congress-text',
 'session',
 'majority',
 'minority',
 'clerk',
 'weburl',
 'members',
 'member',
 'statedistrict',
 'member-info',
 'namelist',
 'bioguideid',
 'lastname',
 'firstname',
 'middlename',
 'sort-name',
 'suffix',
 'courtesy',
 'prior-congress',
 'official-name',
 'formal-name',
 'party',
 'caucus',
 'state',
 'state-fullname',
 'district',
 'townname',
 'office-building',
 'office-room',
 'office-zip',
 'office-zip-suffix',
 'phone',
 'elected-date',
 'sworn-date',
 'committee-assignments',
 'committee',
 'subcommittee',
 'footnote-ref',
 'footnote',
 'predecessor-info',
 'pred-lastname',
 'pred-firstname',
 'pred-middlename',
 'pred-official-name',
 'pred-formal-name',
 'pred-title',
 'pred-memindex',
 'pred-sort-name',
 'pred-party',
 'pred-vacate-date',
 'pred-footnote-ref',
 'pred-footnote',
 'committees',
 'committee-fullname',
 'ratio',
 'subcommittee-fullname']

We can navigate through the tree. You won't do this in practice, but it's helpful for debugging. In this case, we navigated from the root node (`html`) into the `body` tag, then the `memberdata` tag, then the `members` tag. There are 441 descendents at this level, corresponding to the 435 voting seats and the 6 seats for territories.

In [19]:
house_soup.members.member.next_sibling.next_sibling

<member><statedistrict>AL02</statedistrict><member-info><namelist>Moore, Barry</namelist><bioguideid>M001212</bioguideid><lastname>Moore</lastname><firstname>Barry</firstname><middlename></middlename><sort-name>MOORE,BARRY</sort-name><suffix></suffix><courtesy>Mr.</courtesy><prior-congress>117</prior-congress><official-name>Barry Moore</official-name><formal-name>Mr. Moore of Alabama</formal-name><party>R</party><caucus>R</caucus><state postal-code="AL"><state-fullname>Alabama</state-fullname></state><district>2nd</district><townname>Enterprise</townname><office-building>LHOB</office-building><office-room>1504</office-room><office-zip>20515</office-zip><office-zip-suffix>0102</office-zip-suffix><phone>(202) 225-2901</phone><elected-date date="20221108">November  8, 2022</elected-date><sworn-date date="20230107">January  7, 2023</sworn-date></member-info><committee-assignments><committee comcode="AG00" rank="16"></committee><committee comcode="JU00" rank="17"></committee><subcommittee r

In [20]:
len(house_soup.html.body.memberdata.members)

441

You can also short-cut to the members tag directly rather than navigating down the parent elements.

In [ ]:
len(house_soup.members)

The `.contents` method is great for getting a list of the children below the tag as a list. We can use the `[0]` slice to get the first member and their data in the list. Interestingly, the `<committee-assignments>` tags are currently empty since these have not yet been allocated, but will in the next few weeks.

In [32]:
house_soup.members.contents[75]

<member><statedistrict>CO02</statedistrict><member-info><namelist>Neguse, Joe</namelist><bioguideid>N000191</bioguideid><lastname>Neguse</lastname><firstname>Joe</firstname><middlename></middlename><sort-name>NEGUSE,JOE</sort-name><suffix></suffix><courtesy>Mr.</courtesy><prior-congress>117</prior-congress><official-name>Joe Neguse</official-name><formal-name>Mr. Neguse</formal-name><party>D</party><caucus>D</caucus><state postal-code="CO"><state-fullname>Colorado</state-fullname></state><district>2nd</district><townname>Lafayette</townname><office-building>RHOB</office-building><office-room>2400</office-room><office-zip>20515</office-zip><office-zip-suffix>0602</office-zip-suffix><phone>(202) 225-2161</phone><elected-date date="20221108">November  8, 2022</elected-date><sworn-date date="20230107">January  7, 2023</sworn-date></member-info><committee-assignments><committee comcode="II00" rank="6"></committee><committee comcode="JU00" rank="11"></committee><committee comcode="RU00" rank

You could keep navigating down the tree from here.

In [ ]:
house_soup.members.contents[75]

Note that this navigation method breaks when the tag has a hyphen in it.

In [33]:
house_soup.members.contents[75].state-fullname

NameError: name 'fullname' is not defined

Instead you can use the `.find()` method to handle these hyphenated cases.

In [34]:
house_soup.members.contents[75].find('state-fullname')

<state-fullname>Colorado</state-fullname>

We can access the text inside the tag with `.text`

In [35]:
house_soup.members.contents[75].find('state-fullname').text

'Colorado'

The `.find_all()` method will be your primary tool when working with structured data. The `<party>` tag codes party membership (D=Democratic, R=Republican) for each representative. 

In [36]:
house_soup.find_all('party')

[<party>D</party>,
 <party>R</party>,
 <party>R</party>,
 <party>R</party>,
 <party>R</party>,
 <party>R</party>,
 <party>R</party>,
 <party>D</party>,
 <party>R</party>,
 <party>R</party>,
 <party>R</party>,
 <party>R</party>,
 <party>R</party>,
 <party>R</party>,
 <party>R</party>,
 <party>D</party>,
 <party>D</party>,
 <party>R</party>,
 <party>R</party>,
 <party>D</party>,
 <party>R</party>,
 <party>R</party>,
 <party>R</party>,
 <party>D</party>,
 <party>R</party>,
 <party>D</party>,
 <party>R</party>,
 <party>D</party>,
 <party>D</party>,
 <party>D</party>,
 <party>D</party>,
 <party>D</party>,
 <party>D</party>,
 <party>D</party>,
 <party>R</party>,
 <party>D</party>,
 <party>D</party>,
 <party>D</party>,
 <party>D</party>,
 <party>D</party>,
 <party>D</party>,
 <party>R</party>,
 <party>D</party>,
 <party>R</party>,
 <party>R</party>,
 <party>D</party>,
 <party>D</party>,
 <party>D</party>,
 <party>R</party>,
 <party>D</party>,
 <party>D</party>,
 <party>D</party>,
 <party>D</p

There [should be](https://en.wikipedia.org/wiki/118th_United_States_Congress#Party_summary) 211 Democrats and 220 Republicans, plus the other non-voting members from territories.

In [37]:
# Initialize a counter
democrats = 0
republicans = 0
other = 0

# Loop through each element of the caucus tags
for p in house_soup.find_all('party'):
    
    # Check if it's D, R, or something else
    if p.text == "D":
        
        # Increment the appropriate counter
        democrats += 1
    
    elif p.text == "R":
        republicans += 1
        
    else:
        other += 1
        
print("There are {0} Democrats, {1} Republicans, and {2} others in the 118th Congress.".format(democrats,republicans,other))


There are 214 Democrats, 223 Republicans, and 4 others in the 118th Congress.


### Exercises

1. How many members serve in "At Large" districts? (There's at least two ways to do this)
2. How many members are women?
3. How many members are not incumbents? (They were not in the prior Congress)

In [40]:
from collections import Counter

In [42]:
# Empty list to store data
first_name = []

# For each name in the list of "firstname" tags
for name in house_soup.find_all('firstname'):

    # Add the text inside the firstname tag
    first_name.append(name.text)

# Count the number names in first_name
Counter(first_name)


# List comprehension style
Counter([name.text for name in house_soup.find_all('firstname')])

Counter({'Mike': 12,
         'John': 11,
         'Michael': 9,
         'David': 7,
         'Robert': 6,
         'Mark': 6,
         'James': 6,
         'Greg': 5,
         'Scott': 5,
         'Eric': 4,
         'Andy': 4,
         'Daniel': 4,
         'Thomas': 4,
         '': 4,
         'Mary': 3,
         'Steve': 3,
         'Debbie': 3,
         'Jared': 3,
         'Kevin': 3,
         'Tom': 3,
         'Josh': 3,
         'Jim': 3,
         'Brad': 3,
         'Pete': 3,
         'Joe': 3,
         'Bill': 3,
         'Brian': 3,
         'Richard': 3,
         'Andrew': 3,
         'Frank': 3,
         'Jake': 3,
         'Troy': 3,
         'Glenn': 3,
         'Donald': 3,
         'Dan': 3,
         'Jennifer': 3,
         'Barry': 2,
         'J.': 2,
         'Juan': 2,
         'Paul': 2,
         'Doug': 2,
         'Nancy': 2,
         'Jimmy': 2,
         'Julia': 2,
         'Tony': 2,
         'Adam': 2,
         'Grace': 2,
         'Michelle': 2,
        

In [44]:
house_soup.members.contents[0]

<member><statedistrict>AK00</statedistrict><member-info><namelist>Peltola, Mary</namelist><bioguideid>P000619</bioguideid><lastname>Peltola</lastname><firstname>Mary</firstname><middlename>Sattler</middlename><sort-name>PELTOLA,MARY</sort-name><suffix></suffix><courtesy>Mrs.</courtesy><prior-congress>117</prior-congress><official-name>Mary Sattler Peltola</official-name><formal-name>Mrs. Peltola</formal-name><party>D</party><caucus>D</caucus><state postal-code="AK"><state-fullname>Alaska</state-fullname></state><district>At Large</district><townname>Bethel</townname><office-building>CHOB</office-building><office-room>153</office-room><office-zip>20515</office-zip><office-zip-suffix>0200</office-zip-suffix><phone>(202) 225-5765</phone><elected-date date="20221108">November  8, 2022</elected-date><sworn-date date="20230107">January  7, 2023</sworn-date></member-info><committee-assignments><committee comcode="II00" rank="11"></committee><committee comcode="PW00" rank="24"></committee><sub

In [43]:
house_soup.members.contents[75]

<member><statedistrict>CO02</statedistrict><member-info><namelist>Neguse, Joe</namelist><bioguideid>N000191</bioguideid><lastname>Neguse</lastname><firstname>Joe</firstname><middlename></middlename><sort-name>NEGUSE,JOE</sort-name><suffix></suffix><courtesy>Mr.</courtesy><prior-congress>117</prior-congress><official-name>Joe Neguse</official-name><formal-name>Mr. Neguse</formal-name><party>D</party><caucus>D</caucus><state postal-code="CO"><state-fullname>Colorado</state-fullname></state><district>2nd</district><townname>Lafayette</townname><office-building>RHOB</office-building><office-room>2400</office-room><office-zip>20515</office-zip><office-zip-suffix>0602</office-zip-suffix><phone>(202) 225-2161</phone><elected-date date="20221108">November  8, 2022</elected-date><sworn-date date="20230107">January  7, 2023</sworn-date></member-info><committee-assignments><committee comcode="II00" rank="6"></committee><committee comcode="JU00" rank="11"></committee><committee comcode="RU00" rank

In [48]:
# At-Large members
at_large = 0 
for district in house_soup.find_all('statedistrict'):
    if '00' in district.text:
        at_large += 1
at_large

12

In [49]:
house_soup.find_all('district')

[<district>At Large</district>,
 <district>1st</district>,
 <district>2nd</district>,
 <district>3rd</district>,
 <district>4th</district>,
 <district>5th</district>,
 <district>6th</district>,
 <district>7th</district>,
 <district>1st</district>,
 <district>2nd</district>,
 <district>3rd</district>,
 <district>4th</district>,
 <district>Delegate</district>,
 <district>1st</district>,
 <district>2nd</district>,
 <district>3rd</district>,
 <district>4th</district>,
 <district>5th</district>,
 <district>6th</district>,
 <district>7th</district>,
 <district>8th</district>,
 <district>9th</district>,
 <district>1st</district>,
 <district>2nd</district>,
 <district>3rd</district>,
 <district>4th</district>,
 <district>5th</district>,
 <district>6th</district>,
 <district>7th</district>,
 <district>8th</district>,
 <district>9th</district>,
 <district>10th</district>,
 <district>11th</district>,
 <district>12th</district>,
 <district>13th</district>,
 <district>14th</district>,
 <district>15

In [ ]:
at_large = 0 
for district in house_soup.find_all('district'):
    if district.text == 'At Large':
        at_large += 1

at_large

In [52]:
# Women members
women_counter = 0

for i in house_soup.find_all('courtesy'):
    if i.text == "Mrs." or i.text == "Ms.":
        women_counter += 1

women_counter

129

In [53]:
man_counter = 0

for i in house_soup.find_all('courtesy'):
    if i.text == "Mr.":
        man_counter += 1

man_counter

308

In [55]:
Counter([courtesy.text for courtesy in house_soup.find_all('courtesy')])

Counter({'Mr.': 308, 'Ms.': 93, 'Mrs.': 36, '': 4})

In [60]:
# Non-incumbents
incumbent_counter = 0

for i in house_soup.find_all('prior-congress'):
    if i.text == '0':
        incumbent_counter += 1

incumbent_counter

84

### JSON

HTML in the wild can be incredibly complicated (which is why we'll dedicate two whole class sessions to it) and XML is verbose and has other ideosyncracies (which is probably why we won't see it after this class). [JavaScript Object Notation](https://www.json.org/) (JSON) is probably the most popular data markup language and is especially ubiquitous when retreiving data from the application programming interfaces (APIs) of popular platforms like Twitter, Reddit, Wikipedia, *etc*.

JSON is attractive for programmers using JavaScript and Python because it can represent a mix of different data types. We need to make a brief digression into Python's fundamental data stuctures in order to understand the contemporary attraction to JSON. Python has a few fundamental data types for representing collections of information:

* **Lists**: This is a basic ordered data structure that can contain strings, ints, and floats.
* **Dictionaries**: This is an unordered data structure containing key-value pairs, like a phonebook.

Here are some basic examples of lists. Note that lists can contain other lists, and lists of lists of lists, and so on. In this case `school_district` is a list of schools, where each school is a list of classrooms, where each classroom is a list of names.

In [61]:
# Make classrooms as lists with student names as strings
classroom0 = ['Alice','Bob','Carol','Dave']
classroom1 = ['Eve','Frank','Grace','Harold']
classroom2 = ['Isabel','Jack','Katy','Lloyd']
classroom3 = ['Maria','Nate','Olivia','Philip']
classroom4 = ['Quinn','Rachel','Steve','Terry','Ursula']
classroom5 = ['Violet','Walter','Xavier','Yves','Zoe']

# Make schools that contain classrooms
school0 = [classroom0,classroom1]
school1 = [classroom2,classroom3]
school2 = [classroom4,classroom5]

# Make a school district that contains schools
school_district = [school0,school1,school2]

# Inspect one school's enrollments
school_district

[[['Alice', 'Bob', 'Carol', 'Dave'], ['Eve', 'Frank', 'Grace', 'Harold']],
 [['Isabel', 'Jack', 'Katy', 'Lloyd'], ['Maria', 'Nate', 'Olivia', 'Philip']],
 [['Quinn', 'Rachel', 'Steve', 'Terry', 'Ursula'],
  ['Violet', 'Walter', 'Xavier', 'Yves', 'Zoe']]]

You can access list elements by their position. In this case, we could access the first name in `classroom0` by requesting the item at the 0th index.

In [64]:
classroom0[-1]

'Dave'

In [66]:
school_district[0][1][0]

'Eve'

Here are some basic examples of dictionaries. Note that dictionaries can contain other dictionaries, and so on. In this example `mountain_west` is a dictionary of dictionaries where the outer dictionary is keyed by the name of the state and the inner dictionary has different statistical information.

In [67]:
# Make a dictionary of states containing dictionaries
mountain_west = {
    'Colorado': {
        'Abbreviation': 'CO',
        'Area': 269601,
        'Capital': 'Denver',
        'Established': '1876-08-01',
        'Largest city': 'Denver',
        'Population': 5540545,
        'Representatives': 8
    },
    'Idaho': {
        'Abbreviation': 'ID',
        'Area': 216443,
        'Capital': 'Boise',
        'Established': '1890-07-03',
        'Largest city': 'Boise',
        'Population': 1683140,
        'Representatives': 2
    },
    'Montana': {
        'Abbreviation': 'MT',
        'Area': 380831,
        'Capital': 'Helena',
        'Established': '1889-11-08',
        'Largest city': 'Billings',
        'Population': 1042520,
        'Representatives': 2
    },
    'Utah': {
        'Abbreviation': 'UT',
        'Area': 219882,
        'Capital': 'Salt Lake City',
        'Established': '1896-01-04',
        'Largest city': 'Salt Lake City',
        'Population': 3051217,
        'Representatives': 4
    },
    'Wyoming': {
        'Abbreviation': 'WY',
        'Area': 253335,
        'Capital': 'Cheyenne',
        'Established': '1890-07-10',
        'Largest city': 'Cheyenne',
        'Population': 585501,
        'Representatives': 1
    }
}

What are the keys and what are the values?

In [68]:
mountain_west.keys()

dict_keys(['Colorado', 'Idaho', 'Montana', 'Utah', 'Wyoming'])

In [69]:
mountain_west.values()

dict_values([{'Abbreviation': 'CO', 'Area': 269601, 'Capital': 'Denver', 'Established': '1876-08-01', 'Largest city': 'Denver', 'Population': 5540545, 'Representatives': 8}, {'Abbreviation': 'ID', 'Area': 216443, 'Capital': 'Boise', 'Established': '1890-07-03', 'Largest city': 'Boise', 'Population': 1683140, 'Representatives': 2}, {'Abbreviation': 'MT', 'Area': 380831, 'Capital': 'Helena', 'Established': '1889-11-08', 'Largest city': 'Billings', 'Population': 1042520, 'Representatives': 2}, {'Abbreviation': 'UT', 'Area': 219882, 'Capital': 'Salt Lake City', 'Established': '1896-01-04', 'Largest city': 'Salt Lake City', 'Population': 3051217, 'Representatives': 4}, {'Abbreviation': 'WY', 'Area': 253335, 'Capital': 'Cheyenne', 'Established': '1890-07-10', 'Largest city': 'Cheyenne', 'Population': 585501, 'Representatives': 1}])

You access dictionary values by their keys.

In [75]:
# Statistics about Colorado
mountain_west['Colorado']

{'Abbreviation': 'CO',
 'Area': 269601,
 'Capital': 'Denver',
 'Established': '1876-08-01',
 'Largest city': 'Denver',
 'Population': 5540545,
 'Representatives': 8}

What is the population of Colorado? In a dictionary of dictionaries, you can access the inner dictionary with another key.

In [80]:
mountain_west['Colorado']['Population']

5540545

Nested data structures do not need to be the same type. A dictionary can have a list as a value for example. This would probably make the `school_district` example easier to understand.

In [76]:
school_district_dict = {
    'School 1':{
        'Classroom A':['Alice','Bob','Carol','Dave'],
        'Classroom B':['Eve','Frank','Grace','Harold']
    },
    'School 2':{
        'Classroom A':['Isabel','Jack','Katy','Lloyd'],
        'Classroom B':['Maria','Nate','Olivia','Philip']
    },
    'School 3':{
        'Classroom A':['Quinn','Rachel','Steve','Terry','Ursula'],
        'Classroom B':['Violet','Walter','Xavier','Yves','Zoe']
    }
}

We can access the 3rd member (remember Python indexes start at 0, so we ask for 2) of "Classroom B" in "School 2" by working our way outside inward.

In [77]:
school_district_dict['School 2']['Classroom B'][2]

'Olivia'

A list can take just about any other data structure as a value. Dictionaries are great because they label your data for you but don't preserve order and need unique keys, so they're bad at storing repeated data like a time series. Here's a list where the values are dictionaries and each dictionary is information about the population, state, and year of a location.

In [78]:
state_population = [ 
    {'Population': 4338785, 'State': 'Colorado', 'Year': 2000},
    {'Population': 4444513, 'State': 'Colorado', 'Year': 2001},
    {'Population': 4504709, 'State': 'Colorado', 'Year': 2002},
    {'Population': 4555084, 'State': 'Colorado', 'Year': 2003},
    {'Population': 4608811, 'State': 'Colorado', 'Year': 2004},
    {'Population': 4662534, 'State': 'Colorado', 'Year': 2005},
    {'Population': 4745660, 'State': 'Colorado', 'Year': 2006},
    {'Population': 4821784, 'State': 'Colorado', 'Year': 2007},
    {'Population': 4901938, 'State': 'Colorado', 'Year': 2008},
    {'Population': 4976853, 'State': 'Colorado', 'Year': 2009},
    {'Population': 5049935, 'State': 'Colorado', 'Year': 2010},
    {'Population': 5119538, 'State': 'Colorado', 'Year': 2011},
    {'Population': 5191086, 'State': 'Colorado', 'Year': 2012},
    {'Population': 5268413, 'State': 'Colorado', 'Year': 2013},
    {'Population': 5350118, 'State': 'Colorado', 'Year': 2014},
    {'Population': 5448055, 'State': 'Colorado', 'Year': 2015},
    {'Population': 5538180, 'State': 'Colorado', 'Year': 2016}
]

What's the population of Colorado in 2008?

It's actually a small pain to get it out in this list of dictionaries format. You could loop through each dictionary in the list and ask if its "Year" in 2008 and only then print out its "Population". You could discover the min and max years (assuming the data is reliably chronological) and then calculate the index where 2008 should occur. This format probably starts to remind you of some of the limitations of XML, but it has strengths in some situations.

In [81]:
# Loop through each dictionary in the list
for year_data in state_population:
    
    # If the inner dictionary's value for "Year" is 2008, do something, otherwise skip
    if year_data['Year'] == 2008:
        
        # Print our the inner dictionary's value for "Population"
        print(year_data['Population'])

4901938


In [82]:
# Year 2000 is at index 0, so year 2008 should be at index 8
# Passing only the index 8 returns a dictionary
# Include the key whose value you want from this dictionary: "Population"
state_population[8]['Population']

4901938

### Exercises

Below is an example of a [tweet status](https://dev.twitter.com/overview/api/tweets) object that Twitter's [API returns](https://dev.twitter.com/rest/reference/get/statuses/show/id). This `obama_tweet` dictionary corresponds to [this tweet](https://twitter.com/BarackObama/status/831527113211645959). This is a classic example of a JSON object containing a mixture of dictionaries, lists, lists of dictionaries, dictionaries of lists, *etc*.

In [83]:
obama_tweet = {
    'created_at': 'Tue Feb 14 15:34:47 +0000 2017',
    'favorite_count': 1023379,
    'hashtags': [],
    'id': 831527113211645959,
    'id_str': '831527113211645959',
    'lang': 'en',
    'media': [
        {
            'display_url': 'pic.twitter.com/O0UhJWoqGN',
            'expanded_url': 'https://twitter.com/BarackObama/status/831527113211645959/photo/1',
            'id': 831526916398149634,
            'media_url': 'http://pbs.twimg.com/media/C4otUykWcAIbSy1.jpg',
            'media_url_https': 'https://pbs.twimg.com/media/C4otUykWcAIbSy1.jpg',
            'sizes': {
                'large': {'h': 800, 'resize': 'fit', 'w': 1200},
                'medium': {'h': 800, 'resize': 'fit', 'w': 1200},
                'small': {'h': 453, 'resize': 'fit', 'w': 680},
                'thumb': {'h': 150, 'resize': 'crop', 'w': 150}
            },
         'type': 'photo',
         'url': 'https://t.co/O0UhJWoqGN'
        }
    ],
    'retweet_count': 252266,
    'source': '<a href="http://twitter.com" rel="nofollow">Twitter Web Client</a>',
    'text': 'Happy Valentine’s Day, @michelleobama! Almost 28 years with you, but it always feels new. https://t.co/O0UhJWoqGN',
    'urls': [],
    'user': {
        'created_at': 'Mon Mar 05 22:08:25 +0000 2007',
        'description': 'Dad, husband, President, citizen.',
        'favourites_count': 10,
        'followers_count': 84814791,
        'following': True,
        'friends_count': 631357,
        'id': 813286,
        'lang': 'en',
        'listed_count': 221906,
        'location': 'Washington, DC',
        'name': 'Barack Obama',
        'profile_background_color': '77B0DC',
        'profile_background_image_url': 'http://pbs.twimg.com/profile_background_images/451819093436268544/kLbRvwBg.png',
        'profile_banner_url': 'https://pbs.twimg.com/profile_banners/813286/1484945688',
        'profile_image_url': 'http://pbs.twimg.com/profile_images/822547732376207360/5g0FC8XX_normal.jpg',
        'profile_link_color': '2574AD',
        'profile_sidebar_fill_color': 'C2E0F6',
        'profile_text_color': '333333',
        'screen_name': 'BarackObama',
        'statuses_count': 15436,
        'time_zone': 'Eastern Time (US & Canada)',
        'url': 'https://t.co/93Y27HEnnX',
        'utc_offset': -18000,
        'verified': True},
    'user_mentions': [
        {'id': 409486555,
         'name': 'Michelle Obama',
         'screen_name': 'MichelleObama'}
    ]
}

1. What are the top-most keys in the `obama_tweet` object?
2. When was this tweet sent?
3. Does this tweet mention anyone?
4. How many retweets did this tweet receive (at the time I collected it)?
5. How many followers does the "user" who wrote this tweet have?
6. What's the "media_url" for the image in this tweet?

In [84]:
# What are keys?
obama_tweet.keys()

dict_keys(['created_at', 'favorite_count', 'hashtags', 'id', 'id_str', 'lang', 'media', 'retweet_count', 'source', 'text', 'urls', 'user', 'user_mentions'])

In [85]:
# When sent?
obama_tweet['created_at']

'Tue Feb 14 15:34:47 +0000 2017'

In [88]:
# Who mentioned?
obama_tweet['user_mentions'][0]['screen_name']

'MichelleObama'

In [92]:
obama_tweet['text']

'Happy Valentine’s Day, @michelleobama! Almost 28 years with you, but it always feels new. https://t.co/O0UhJWoqGN'

In [95]:
import re

re.findall(r'\@(\w+)','@bkeegan_1234 is a cool dude')

['bkeegan_1234']

In [96]:
# How many retweets?
obama_tweet['retweet_count']

252266

In [100]:
# How many followers?
obama_tweet['user']#['followers_count']

{'created_at': 'Mon Mar 05 22:08:25 +0000 2007',
 'description': 'Dad, husband, President, citizen.',
 'favourites_count': 10,
 'followers_count': 84814791,
 'following': True,
 'friends_count': 631357,
 'id': 813286,
 'lang': 'en',
 'listed_count': 221906,
 'location': 'Washington, DC',
 'name': 'Barack Obama',
 'profile_background_color': '77B0DC',
 'profile_background_image_url': 'http://pbs.twimg.com/profile_background_images/451819093436268544/kLbRvwBg.png',
 'profile_banner_url': 'https://pbs.twimg.com/profile_banners/813286/1484945688',
 'profile_image_url': 'http://pbs.twimg.com/profile_images/822547732376207360/5g0FC8XX_normal.jpg',
 'profile_link_color': '2574AD',
 'profile_sidebar_fill_color': 'C2E0F6',
 'profile_text_color': '333333',
 'screen_name': 'BarackObama',
 'statuses_count': 15436,
 'time_zone': 'Eastern Time (US & Canada)',
 'url': 'https://t.co/93Y27HEnnX',
 'utc_offset': -18000,
 'verified': True}

In [ ]:
# What's media URL?

## Show-and-tell

For Thursday, come to class with examples of the following for show-and-tell.

### Parse the XML of an RSS feed

[Really Simple Syndication](https://en.wikipedia.org/wiki/RSS) (RSS) is a format for serving webfeeds commonly used for blogs and for podcasts.

1. Find the RSS feed for a blog, podcast, or some other online property. For example, *This American Life* hosts its RSS feed [here](http://feed.thisamericanlife.org/talpodcast).
2. Use `requests` to import the RSS feed into your notebook.
3. Review the ontology of the RSS. What are some interesting objects?
4. Use `BeautifulSoup` to parse the raw XML.
5. Do some kind of exploratory data analysis.

In [145]:
tam_raw = requests.get('https://www.thisamericanlife.org/podcast/rss.xml').text
tam_raw[:1000]

'<?xml version="1.0" encoding="utf-8"?>\n<rss version="2.0" xml:base="https://www.thisamericanlife.org"  xmlns:dc="http://purl.org/dc/elements/1.1/" xmlns:itunes="http://www.itunes.com/dtds/podcast-1.0.dtd">\n<channel>\n <title>This American Life</title>\n <link>https://www.thisamericanlife.org</link>\n <description>Each week we choose a theme. Then anything can happen. This American Life is true stories that unfold like little movies for radio. Personal stories with funny moments, big feelings, and surprising plot twists. Newsy stories that try to capture what it’s like to be alive right now. It’s the most popular weekly podcast in the world, and winner of the first ever Pulitzer Prize for a radio show or podcast. Hosted by Ira Glass and produced in collaboration with WBEZ Chicago.</description>\n <language>en</language>\n <itunes:new-feed-url>https://www.thisamericanlife.org/podcast/rss.xml</itunes:new-feed-url>\n <copyright>Copyright 1995-2024 This American Life</copyright>\n <itune

In [146]:
tam_soup = BeautifulSoup(tam_raw,features='xml')

In [149]:
tam_soup.contents[0]

<rss version="2.0" xml:base="https://www.thisamericanlife.org" xmlns:dc="http://purl.org/dc/elements/1.1/" xmlns:itunes="http://www.itunes.com/dtds/podcast-1.0.dtd">
<channel>
<title>This American Life</title>
<link>https://www.thisamericanlife.org</link>
<description>Each week we choose a theme. Then anything can happen. This American Life is true stories that unfold like little movies for radio. Personal stories with funny moments, big feelings, and surprising plot twists. Newsy stories that try to capture what it’s like to be alive right now. It’s the most popular weekly podcast in the world, and winner of the first ever Pulitzer Prize for a radio show or podcast. Hosted by Ira Glass and produced in collaboration with WBEZ Chicago.</description>
<language>en</language>
<itunes:new-feed-url>https://www.thisamericanlife.org/podcast/rss.xml</itunes:new-feed-url>
<copyright>Copyright 1995-2024 This American Life</copyright>
<itunes:author>This American Life</itunes:author>
<itunes:subti

In [128]:
[tag.text for tag in tam_soup.find_all('pubDate')]

['Fri, 07 Jun 2024 12:14:53 PST',
 'Fri, 07 Jun 2024 12:14:53 PST',
 'Sun, 19 Nov 2023 12:14:53 PST',
 'Sun, 15 Jan 2023 15:20:53 PST',
 'Sun, 6 Mar 2022 19:08:44 PST',
 'Tue, 8 Jun 2021 15:20:35 PST',
 'Fri, 13 Nov 2020 17:08:26 PST',
 'Wed, 3 Jun 2020 15:20:44 PST',
 'Thu, 24 Oct 2019 16:10:53 PST',
 'Sat, 12 Jan 2019 10:07:44 PST',
 'Sat, 14 Jul 2018 12:05:35 PST',
 'Sat, 27 Jan 2018 23:58:44 PST',
 'Wed, 09 Aug 2017 15:20:53 PST',
 'Tue, 24 Jan 2017 21:05:35 PST',
 'Sun, 07 Aug 2016 12:05:26 PST',
 'Sun, 20 Mar 2016 00:08:08 PST',
 'Wed, 28 Oct 2015 17:35:08 PST']

### Parse the JSON returned from Wikipedia

Wikipedia serves up data about its articles from its API ([docs](https://www.mediawiki.org/wiki/API:Revisions)) using JSON. `wikipedia_500_recent_revisions` a simple function that will retrieve the 500 most recent revisions for an English Wikipedia article.

1. Use the function to retrieve the most recent revisions for an English Wikipedia article.
2. Navigate to the part of JSON document where the "revisions" lists lives.
3. Count the number of revisions in the list. If you have 500 revisions, come up with a hypothesis about what the "rvcontinue" parameter might mean.
4. Find the user who made the most contributions in your data.

In [129]:
def wikipedia_500_recent_revisions(page_title):    
    
    query_params = {
        'action':'query',
        'titles':page_title,
        'prop':'revisions',
        'rvprop':'ids|userid|comment|timestamp|user|size|sha1',
        'rvlimit':500,
        'rvdir':'older',
        'format':'json'
    }
    
    return requests.get(url = "https://en.wikipedia.org/w/api.php", params = query_params).json()

In [138]:
revs = wikipedia_500_recent_revisions('Deion Sanders')

In [143]:
import pandas as pd

In [144]:
pd.DataFrame(revs['query']['pages']['901141']['revisions'])

,revid,parentid,user,userid,timestamp,size,sha1,comment,anon
0,1245092889,1244784249,GreenC bot,27823944,2024-09-11T00:00:01Z,106157,619277708ce3869d80c7dce92dbb7622cb4ddf4b,Move 1 url. [[User:GreenC/WaybackMedic_2.5|Way...,NaN
1,1244784249,1244615639,Bluerules,5294038,2024-09-09T04:59:21Z,106198,7c9336bc431871b48bb9e1f7004456cfc2a68907,Redundant.,NaN
2,1244615639,1244615601,Erik255,18210084,2024-09-08T03:16:31Z,106240,e076b5440d8d89b5e4721ecddaf9e694d99363fc,,NaN
3,1244615601,1244559356,Erik255,18210084,2024-09-08T03:16:11Z,106240,8e343646d55c483162ed4364f6b07b64b8044a28,/* Head coaching record */Updated record,NaN
4,1244559356,1244390045,108.41.83.160,0,2024-09-07T20:44:03Z,106240,da415d8391e0f6030bf6808cfd74042f37cfdd0a,Fixed,
...,...,...,...,...,...,...,...,...,...
495,1157795061,1157794997,Marist72,31096180,2023-05-31T01:15:10Z,88279,5a6e6f728f524bb2a6cf225c2eae9329264b8dec,,NaN
496,1157794997,1157794954,Marist72,31096180,2023-05-31T01:14:37Z,88246,2047a0326f97dbbb35a5b144d4ff6fdcbb45a00f,,NaN
497,1157794954,1157794822,Marist72,31096180,2023-05-31T01:14:10Z,88253,b252fd57ae3f2ac8e6aaef156658723ba9b38559,,NaN
498,1157794822,1157794740,Marist72,31096180,2023-05-31T01:12:44Z,88286,5a48775d78da6335c15e980a1f8cb0ce8ec78a82,,NaN
